# balance: outcome modelling end-to-end (the g-computation estimate `μ̂_OM`)

Once a sample has been reweighted to a target, `balance` can estimate the target-population mean of an **outcome** in more than one way. This tutorial walks the outcome-model (g-computation) estimator `μ̂_OM` end-to-end and compares it to the inverse-propensity-weighted estimate `μ̂_IPW`, on the same simulated data as the [Quickstart](./quickstart).

- **`μ̂_IPW` — inverse-propensity (Hájek) weighted mean** (`outcomes().mean()`): the weighted average of the responders' *observed* outcome using the adjustment weights. Consistent when the **weighting** (propensity) model is correct.
- **`μ̂_OM` — outcome-model / g-computation estimate** (`outcomes_hat().mean()`): fit a learner `ĝ(X) ≈ E[Y|X]` on the responders, apply it to the **target** covariates, and average the predicted outcomes with the target weights. Consistent when the **outcome** model is correct. Added in v0.23.

The two estimators use the covariates differently, so comparing them is a useful robustness check: agreement is reassuring, disagreement points at a misspecified weighting or outcome model. (A doubly-robust / AIPW estimator that combines both is planned for a later release.)

## Load the data

We use the same simulated toy dataset as the Quickstart: a biased `sample` and the `target` population we want to estimate. The outcome is `happiness`; the target also carries `happiness` here only so we can compare our estimates against the ground truth.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from balance import load_data, Sample

target_df, sample_df = load_data()
sample_df.head()

## Build a `Sample`, link the target, and adjust

We load the sample and target into `Sample` objects (declaring `happiness` as the outcome), link them, and run the default IPW adjustment — exactly the Quickstart workflow. The outcome model itself does **not** require the weights, but we adjust here so we can compare `μ̂_OM` against `μ̂_IPW`.

In [ ]:
sample = Sample.from_frame(sample_df, outcome_columns=["happiness"])
target = Sample.from_frame(target_df, outcome_columns=["happiness"])

bf = sample.set_target(target).adjust(method="ipw")

## `μ̂_IPW` — the inverse-propensity weighted estimate

`outcomes().mean()` averages the **observed** outcome across sources:

- the **`self`** row is the *reweighted sample* — this is `μ̂_IPW`, the IPW estimate of the target mean;
- the **`target`** row is the target's own observed `happiness` (the ground truth, available in this simulation);
- the **`unadjusted`** row is the raw (unweighted) sample mean.

The unadjusted sample badly under-estimates the target; reweighting pulls the `self` estimate toward the truth.

In [ ]:
bf.outcomes().mean()

## `μ̂_OM` — the outcome-model (g-computation) estimate

Now the outcome model. `fit_outcome_model()` fits `ĝ(X) ≈ E[Y|X]` on the responders (the default `model="auto"` is a gradient-boosted tree that handles the categorical covariates natively). `predict_outcomes(on="both")` then scores **both** the responders (in-sample `ŷ`) and the **target**, and populates the `outcomes_hat` (`ŷ`) columns.

In `outcomes_hat().mean()` the **`target`** row is `μ̂_OM`: the weighted mean of the predicted outcome over the target.

In [ ]:
bf.fit_outcome_model(model="auto")
bf.predict_outcomes(on="both")

bf.outcomes_hat().mean()

## An honest confidence interval for `μ̂_OM`

The analytic CI treats the predictions `ŷ` as fixed, so it *under-covers* `μ̂_OM`. `mean_with_ci(ci_method="bootstrap")` gives an honest interval: it resamples the responders, refits `ĝ*`, predicts on the **fixed** target, and re-averages — a percentile CI over the replicates. It is deterministic given `random_seed`.

In [ ]:
bf.outcomes_hat().mean_with_ci(ci_method="bootstrap", n_bootstrap=200, random_seed=2020)

`outcomes_hat().summary()` reports the estimator and *scopes* any doubly-robust claim to the fit weights — it never prints a blanket "doubly robust" (a plain g-computation for the non-linear default; doubly robust only for a weighted linear-with-intercept fit).

In [ ]:
print(bf.outcomes_hat().summary())

## IPW vs. outcome model, side by side

Both estimators substantially reduce the bias of the raw sample mean and land close to the ground truth — and close to each other, which is the reassuring case. A large gap between `μ̂_IPW` and `μ̂_OM` would instead flag a misspecified weighting or outcome model.

In [ ]:
import pandas as pd

ipw = bf.outcomes().mean()
om = bf.outcomes_hat().mean()
pd.DataFrame(
    {
        "estimate_of_target_happiness": {
            "unadjusted (raw sample mean)": ipw.loc["unadjusted", "happiness"],
            "mu_hat_IPW (reweighted sample)": ipw.loc["self", "happiness"],
            "mu_hat_OM (outcome model)": om.loc["target", "happiness_hat"],
            "ground truth (target)": ipw.loc["target", "happiness"],
        }
    }
).round(2)

## Train / holdout transfer

You can fit the outcome model on one frame (a *train* split) and apply it to a *different* frame (a *holdout* / scoring split) with the same covariate schema — mirroring how `set_fitted_model` transfers a fitted IPW model. `set_fitted_outcome_model` copies the already-fitted model onto the holdout **without re-fitting** (the fitted learner is shared by identity), so predicting on the holdout target gives `μ̂_OM` computed with the train model.

In [ ]:
half = len(sample_df) // 2
train_bf = (
    Sample.from_frame(sample_df.iloc[:half], outcome_columns=["happiness"])
    .set_target(target)
    .adjust(method="ipw")
)
train_bf.fit_outcome_model(model="auto")

holdout_bf = Sample.from_frame(
    sample_df.iloc[half:], outcome_columns=["happiness"]
).set_target(target)

# Graft the train model onto the holdout (no re-fitting), then score the holdout.
scored = holdout_bf.set_fitted_outcome_model(train_bf, inplace=False)
scored.predict_outcomes(on="both")
scored.outcomes_hat().mean()  # target row = mu_hat_OM on the holdout target

## Choosing the model (`model=`) and its inputs (`variables=`)

The estimator is chosen with `model=` and the covariate inputs with `variables=`.
`model=` accepts `"auto"` (a `HistGradientBoosting` regressor/classifier picked by
outcome type), a single sklearn estimator (cloned per outcome), a
`{"_discrete": clf, "_continuous": reg}` **type map**, or a
`{outcome_column: estimator}` **column map**. `variables=` restricts the model inputs
`X` to a subset of the covariates (default: all of them).

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression

# a single sklearn estimator (cloned per outcome), restricted to a covariate subset:
bf.fit_outcome_model(model=LinearRegression(), variables=["age_group", "income"])
bf.outcome_model["X_matrix_columns"][:4]

### Weighted fit and the doubly-robust special case

The outcome model is **unweighted by default** (`weighted=False`): it estimates
`E[Y|X]` and is usually best left unbiased by the design weights. Passing
`weighted=True` fits with the frame's active weights; for a **linear model with an
intercept**, a weighted-least-squares fit makes the estimate doubly robust with
respect to those weights, which `summary()` reports explicitly.

In [ ]:
bf.fit_outcome_model(model=LinearRegression(), weighted=True)
bf.predict_outcomes(on="both")
print(bf.outcomes_hat().summary())

## Binary outcomes: classification, calibration, and a per-type model map

A binary 0/1 outcome is modelled with a classifier and stored as the predicted
probability `P(Y=1)` (so the estimate is the target prevalence). `calibrate=True`
wraps the classifier in `CalibratedClassifierCV`. Below, a `{"_discrete",
"_continuous"}` type map routes the binary `happy` outcome to logistic regression.

In [ ]:
sample_bin = Sample.from_frame(
    sample_df.assign(
        happy=(sample_df["happiness"] > sample_df["happiness"].median()).astype(int)
    ).drop(columns=["happiness"]),
    outcome_columns=["happy"],
)
sample_bin.fit_outcome_model(
    model={"_discrete": LogisticRegression(max_iter=1000), "_continuous": LinearRegression()},
    calibrate=True,
)
sample_bin.outcome_model["prediction_kind"], sample_bin.outcome_model["calibrated"]

## Fit and predict in one call

`fit_predict_outcomes(...)` is the sklearn-style `fit_predict`: it fits the model and
then writes the `<outcome>_hat` columns in a single call.

In [ ]:
bf.fit_predict_outcomes(model="auto", on="both")
bf.outcomes_hat().mean()

## Under the hood: the pure DataFrame API

The frame methods above wrap pure functions in `balance.outcome_models` that operate
on plain DataFrames: `fit_outcome_model` / `predict_outcome`, the reusable
`bootstrap_outcome_estimate` engine, and `learner_from_model` (which reconstructs the
per-outcome estimators). `weighted_r2` (in `stats_and_plots`) is the weighted
regression-fit metric behind the stored `perf`, and `add_outcomes_hat_column` attaches
a `<outcome>_hat` column directly.

In [ ]:
from balance.outcome_models import (
    fit_outcome_model,
    predict_outcome,
    learner_from_model,
    bootstrap_outcome_estimate,
)
from balance.stats_and_plots.weighted_stats import weighted_r2

covars_R, y_R = sample.covars().df, sample.outcomes().df
w_R = sample.weights().df.iloc[:, 0]

model = fit_outcome_model(covars_R, y_R, sample_weight=w_R, model="auto")  # pure fit -> dict
yhat_R = predict_outcome(model, covars_R)["happiness"]                     # pure replay
print("weighted R2:", round(weighted_r2(y_R["happiness"], yhat_R, w=w_R), 3))
print("reconstructed learners:", list(learner_from_model(model)))
print("bootstrap:", bootstrap_outcome_estimate(
    covars_R, y_R, w_R, target.covars().df, target.weights().df.iloc[:, 0],
    fit_kwargs={"model": LinearRegression()}, n_bootstrap=200, random_seed=2020,
)["happiness"])

# attach a predicted-outcome column directly (the low-level data-model accessor):
sample.add_outcomes_hat_column("happiness_hat", pd.Series(yhat_R, index=covars_R.index))
sample.outcomes_hat_columns

## Where to go next

- The [Quickstart](./quickstart) covers the reweighting workflow (diagnostics, adjustment methods, weights) that produces the `μ̂_IPW` estimate above.
- `outcomes().weights_impact_on_outcome_ss()` is a *diagnostic* (not a third estimator): it reports how much the weights move the outcome mean, with a significance test.
- The [outcome-model design doc](https://github.com/facebookresearch/balance/blob/main/docs/architecture/architecture_0_23_0.md) covers the estimator theory and the AIPW roadmap.

In [ ]:
import session_info

session_info.show(html=False, dependencies=True)